# Session 09: MCP Servers

## Understanding and Building with the Model Context Protocol

### Learning Objectives

- **Understand the Model Context Protocol (MCP)** — what it is, why it matters, and how it enables AI models to interact with external tools and data sources
- **Consume remote MCP servers** — connect to public MCP servers (GitMCP, GitHub) and use their tools through the Anthropic-powered agent
- **Build a custom MCP server** — create a Stone Ridge investment tools server using `FastMCP`
- **Integrate MCP servers with LangGraph** — use `langchain-mcp-adapters` and `MultiServerMCPClient` to bring MCP tools into a LangGraph agent

### Overview

The **Model Context Protocol (MCP)** is an open standard that defines how AI models connect to external tools and data sources. Think of it as a universal adapter:

- **Without MCP**: Every AI app needs custom integration code for every tool/API
- **With MCP**: Tools expose themselves via a standard protocol; any MCP-compatible client can discover and use them

In this notebook we will:
1. Connect to **remote MCP servers** (GitMCP for documentation, GitHub MCP for repo operations)
2. Build a **custom MCP server** (`mcp_server.py`) with Stone Ridge investment tools
3. Consume our custom server from a **LangGraph agent** via `langchain-mcp-adapters`

---

## Task 1: Dependencies & Environment

In [1]:
import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key:")

# Optional: GitHub PAT for GitHub MCP Server
if not os.environ.get("GITHUB_PAT"):
    pat = getpass.getpass("GitHub PAT (optional — press Enter to skip):")
    if pat.strip():
        os.environ["GITHUB_PAT"] = pat

In [ ]:
# nest_asyncio is NOT needed — IPython 9+ supports top-level await natively.
# Importing it actually breaks anyio/sniffio task detection on Python 3.14.
# If you see "unknown async library" errors, make sure nest_asyncio is NOT applied.

In [5]:
import os
import certifi

# Create a combined cert bundle with Zscaler for corporate network
zscaler_cert = "/Users/jaden.lee/ZscalerRootCertificate-2048-SHA256-Feb2025.crt"
combined_cert = "/Users/jaden.lee/combined_certs.pem"

with open(combined_cert, "w") as outfile:
    with open(certifi.where(), "r") as certifi_file:
        outfile.write(certifi_file.read())
    with open(zscaler_cert, "r") as zscaler_file:
        outfile.write(zscaler_file.read())

os.environ['REQUESTS_CA_BUNDLE'] = combined_cert
os.environ['SSL_CERT_FILE'] = combined_cert
os.environ['CURL_CA_BUNDLE'] = combined_cert

In [7]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="us.anthropic.claude-sonnet-4-20250514-v1:0", temperature=0)

# Test the connection
response = llm.invoke("Say 'MCP server ready!' in exactly those words.")
print(response.content)

MCP server ready!


---

## Task 2: Remote MCP Servers — GitMCP

**GitMCP** turns any GitHub repository's documentation into a searchable MCP server — no authentication required for public repos.

We'll connect to the LangChain documentation via GitMCP to demonstrate the protocol in action.

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect to GitMCP — a public MCP server for GitHub documentation
gitmcp_client = MultiServerMCPClient(
    {
        "langchain_docs": {
            "transport": "http",
            "url": "https://gitmcp.io/langchain-ai/langchain",
        }
    }
)

gitmcp_tools = await gitmcp_client.get_tools()

print(f"Loaded {len(gitmcp_tools)} tools from GitMCP:")
for t in gitmcp_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

AsyncLibraryNotFoundError: unknown async library, or not in async context

### Using GitMCP tools in a LangGraph Agent

Let's wire these tools into a simple ReAct agent and ask a documentation question.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


llm_with_gitmcp = llm.bind_tools(gitmcp_tools)


def agent_node(state: AgentState):
    messages = [
        SystemMessage(content="You are a helpful assistant with access to documentation tools. Use them to answer questions accurately.")
    ] + state["messages"]
    return {"messages": [llm_with_gitmcp.invoke(messages)]}


def should_continue(state: AgentState):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"


workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(gitmcp_tools, handle_tool_errors=True))
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
workflow.add_edge("tools", "agent")

gitmcp_agent = workflow.compile()
print("GitMCP agent compiled!")

In [ ]:
result = gitmcp_agent.invoke(
    {"messages": [HumanMessage(content="How do I create a custom tool in LangChain?")]}
)
print(result["messages"][-1].content[:1500])

---

## Task 3: Remote MCP Servers — GitHub MCP

The **GitHub MCP Server** is an official, GitHub-maintained server that gives agents the ability to manage repositories, issues, pull requests, and more.

> **Requires**: A GitHub Personal Access Token (fine-grained) with `Contents: Read and write`, `Pull requests: Read and write`, and `Metadata: Read-only` permissions.

| MCP Tool | Replaces | What It Does |
|---|---|---|
| `create_repository` | `git init` + GitHub UI | Creates a new repo |
| `create_or_update_file` | `git add` + `git commit` | Commits a file to a branch |
| `create_branch` | `git checkout -b` | Creates a new branch |
| `create_pull_request` | `gh pr create` | Opens a PR |
| `search_repositories` | `gh repo list` | Searches repos |
| `get_file_contents` | `git show` / `cat` | Reads a file |
| `list_commits` | `git log` | Shows commit history |

In [ ]:
if os.environ.get("GITHUB_PAT"):
    github_client = MultiServerMCPClient(
        {
            "github": {
                "transport": "http",
                "url": "https://api.githubcopilot.com/mcp/",
                "headers": {
                    "Authorization": f"Bearer {os.environ['GITHUB_PAT']}",
                },
            }
        }
    )

    github_mcp_tools = await github_client.get_tools()
    print(f"Loaded {len(github_mcp_tools)} GitHub MCP tools:")
    for t in github_mcp_tools:
        print(f"  - {t.name}: {t.description[:80]}...")
else:
    print("GITHUB_PAT not set — skipping GitHub MCP. Set it above to enable.")
    github_mcp_tools = []

---

## Task 4: Build a Custom MCP Server

Now let's look at **building our own MCP server**. The file `mcp_server.py` in this directory implements a Stone Ridge investment tools server using `FastMCP`.

### Server Architecture

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Stone Ridge Investment Tools")

@mcp.tool()
def get_fund_overview(fund_name: str) -> str:
    """Return an overview of a Stone Ridge fund."""
    ...

@mcp.tool()
def get_investment_philosophy() -> str:
    """Return Stone Ridge's investment philosophy."""
    ...

@mcp.tool()
def analyze_portfolio_allocation(...) -> str:
    """Analyze a portfolio from a Stone Ridge perspective."""
    ...

@mcp.tool()
def search_investor_letter(query: str, section: str = "all") -> str:
    """Keyword search across the investor letter."""
    ...

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

### Testing with MCP Inspector

You can test the server interactively:

```bash
uv run mcp dev mcp_server.py
```

This launches the **MCP Inspector** — a web UI where you can call each tool and see the responses.

Let's review the server code:

In [ ]:
# Let's peek at the MCP server code
with open("mcp_server.py") as f:
    print(f.read())

### Quick direct test of the MCP server functions

Since the tools are regular Python functions, we can import and call them directly:

In [ ]:
# Direct import test (no MCP transport needed)
from mcp_server import get_fund_overview, get_investment_philosophy, analyze_portfolio_allocation, search_investor_letter

print("=== Fund Overview: Bitcoin ===")
print(get_fund_overview(fund_name="bitcoin"))
print()

print("=== Portfolio Analysis ===")
print(analyze_portfolio_allocation(
    equities_pct=50.0,
    fixed_income_pct=25.0,
    alternatives_pct=20.0,
    cash_pct=5.0,
))
print()

print("=== Letter Search: 'Bayesian' ===")
print(search_investor_letter(query="Bayesian"))

---

## Task 5: Consume the Custom MCP Server with `langchain-mcp-adapters`

Now we'll connect to our custom MCP server using `MultiServerMCPClient` and use it inside a LangGraph agent. This demonstrates the full MCP lifecycle:

1. **Build** a server (`mcp_server.py`)
2. **Connect** via `langchain-mcp-adapters`
3. **Use** the tools in a LangGraph agent

The `MultiServerMCPClient` supports `stdio` transport — it launches the server as a subprocess and communicates over stdin/stdout.

In [ ]:
import sys

# Connect to our custom MCP server via stdio transport
custom_mcp_client = MultiServerMCPClient(
    {
        "stone_ridge": {
            "transport": "stdio",
            "command": sys.executable,  # Use the current Python interpreter
            "args": ["mcp_server.py"],
        }
    }
)

stone_ridge_tools = await custom_mcp_client.get_tools()

print(f"Loaded {len(stone_ridge_tools)} tools from Stone Ridge MCP server:")
for t in stone_ridge_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

In [ ]:
# Build a LangGraph agent with our custom MCP tools
llm_with_sr = llm.bind_tools(stone_ridge_tools)


def sr_agent_node(state: AgentState):
    messages = [
        SystemMessage(
            content="You are the Stone Ridge Investment Assistant. Use your tools to answer "
            "questions about Stone Ridge funds, investment philosophy, and portfolio analysis."
        )
    ] + state["messages"]
    return {"messages": [llm_with_sr.invoke(messages)]}


sr_workflow = StateGraph(AgentState)
sr_workflow.add_node("agent", sr_agent_node)
sr_workflow.add_node("tools", ToolNode(stone_ridge_tools, handle_tool_errors=True))
sr_workflow.add_edge(START, "agent")
sr_workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
sr_workflow.add_edge("tools", "agent")

sr_agent = sr_workflow.compile()
print("Stone Ridge MCP agent compiled!")

In [ ]:
result = sr_agent.invoke(
    {"messages": [HumanMessage(content="What is Stone Ridge's investment philosophy? How do they think about bitcoin?")]}
)
print(result["messages"][-1].content)

In [ ]:
result = sr_agent.invoke(
    {"messages": [HumanMessage(
        content="Analyze this portfolio: 40% equities, 30% fixed income, 25% alternatives, 5% cash. "
        "What would Stone Ridge say about this allocation?"
    )]}
)
print(result["messages"][-1].content)

---

## Task 6: Multi-Server MCP Agent

The real power of MCP is combining **multiple servers** into a single agent. Let's build an agent that has access to:
- **Stone Ridge tools** (our custom server)
- **GitMCP tools** (documentation search)
- **GitHub MCP tools** (if PAT configured)

In [ ]:
# Combine all available MCP tools
all_mcp_tools = stone_ridge_tools + gitmcp_tools
if github_mcp_tools:
    all_mcp_tools += github_mcp_tools

print(f"Total MCP tools available: {len(all_mcp_tools)}")

llm_multi = llm.bind_tools(all_mcp_tools)


def multi_agent_node(state: AgentState):
    messages = [
        SystemMessage(
            content="You are an investment research assistant with access to Stone Ridge investment tools, "
            "documentation search, and GitHub repository tools. Use the appropriate tools for each query."
        )
    ] + state["messages"]
    return {"messages": [llm_multi.invoke(messages)]}


multi_wf = StateGraph(AgentState)
multi_wf.add_node("agent", multi_agent_node)
multi_wf.add_node("tools", ToolNode(all_mcp_tools, handle_tool_errors=True))
multi_wf.add_edge(START, "agent")
multi_wf.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
multi_wf.add_edge("tools", "agent")

multi_agent = multi_wf.compile()
print("Multi-server MCP agent compiled!")

In [ ]:
result = multi_agent.invoke(
    {"messages": [HumanMessage(
        content="Search the Stone Ridge investor letter for information about reinsurance, "
        "then look up how LangChain implements tool calling."
    )]}
)
print(result["messages"][-1].content[:2000])

### ❓ Question #1

What are the benefits of the MCP protocol over writing custom API wrappers? When would you still prefer a direct `@tool` wrapper instead?

##### Answer

**Benefits of MCP over custom API wrappers:**

- **Standardized discovery**: MCP clients automatically discover all available tools from a server (as we saw — connecting to GitMCP instantly gave us 4 tools, and GitHub MCP exposed 30+). With custom `@tool` wrappers, you must manually write and maintain each one.

- **Separation of concerns**: The tool implementation lives in the MCP server, completely decoupled from the agent code. Teams can develop, version, and deploy servers independently. With `@tool` wrappers, tool logic is embedded directly in the agent's codebase.

- **Composability**: Multiple MCP servers combine naturally into a single agent (as demonstrated in Task 6, where we merged Stone Ridge, GitMCP, and GitHub tools). Each server is a self-contained unit that can be added or removed without changing agent code.

- **Reusability across frameworks**: An MCP server works with any MCP-compatible client — Claude Desktop, LangChain, custom apps. A `@tool` function is tied to LangChain specifically.

- **Community ecosystem**: Official MCP servers (like GitHub's) are maintained by the API provider, keeping pace with API changes automatically.

**When to prefer a direct `@tool` wrapper instead:**

- **Simple APIs with few endpoints**: If you only need 1-3 API calls (like the X API search in the Connectors notebook), building an MCP server adds unnecessary infrastructure overhead.

- **Custom response formatting**: When you need to transform, filter, or truncate API responses for the LLM's context window, a `@tool` gives you full control over the return value.

- **No existing MCP server**: If no one has built an MCP server for the API and you don't need the reusability benefits, a `@tool` is faster to implement (minutes vs. hours).

- **Tight coupling is desirable**: When the tool logic is specific to your agent's use case and unlikely to be reused elsewhere, the simplicity of a decorated function outweighs the protocol overhead.

### \ud83c\udfd7\ufe0f Activity #1

Extend the `mcp_server.py` with a new tool:

1. Add a `compare_funds(fund_a: str, fund_b: str)` tool that returns a side-by-side comparison
2. Restart the MCP server connection
3. Test it through the agent by asking: *"Compare the SRE fund with the Bitcoin strategy"*

In [ ]:
# --- Step 1: compare_funds has been added to mcp_server.py ---
# Quick direct test to verify:
from mcp_server import compare_funds
print(compare_funds(fund_a="SRE", fund_b="Bitcoin"))

# --- Step 2: Restart the MCP server connection ---
import sys

custom_mcp_client_v2 = MultiServerMCPClient(
    {
        "stone_ridge": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["mcp_server.py"],
        }
    }
)

stone_ridge_tools_v2 = await custom_mcp_client_v2.get_tools()

print(f"\nLoaded {len(stone_ridge_tools_v2)} tools (should now include compare_funds):")
for t in stone_ridge_tools_v2:
    print(f"  - {t.name}: {t.description[:70]}...")

# --- Step 3: Rebuild agent and test ---
llm_with_sr_v2 = llm.bind_tools(stone_ridge_tools_v2)


def sr_agent_node_v2(state: AgentState):
    messages = [
        SystemMessage(
            content="You are the Stone Ridge Investment Assistant. Use your tools to answer "
            "questions about Stone Ridge funds, investment philosophy, and portfolio analysis."
        )
    ] + state["messages"]
    return {"messages": [llm_with_sr_v2.invoke(messages)]}


sr_workflow_v2 = StateGraph(AgentState)
sr_workflow_v2.add_node("agent", sr_agent_node_v2)
sr_workflow_v2.add_node("tools", ToolNode(stone_ridge_tools_v2, handle_tool_errors=True))
sr_workflow_v2.add_edge(START, "agent")
sr_workflow_v2.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
sr_workflow_v2.add_edge("tools", "agent")

sr_agent_v2 = sr_workflow_v2.compile()

result = sr_agent_v2.invoke(
    {"messages": [HumanMessage(content="Compare the SRE fund with the Bitcoin strategy")]}
)
print("\n=== Agent Response ===")
print(result["messages"][-1].content)

---

## Summary

In this notebook we covered:

- **MCP Protocol Concepts**: How the Model Context Protocol enables standardized tool discovery and invocation
- **Remote MCP Servers**: Connected to GitMCP (documentation) and GitHub MCP (repo operations)
- **Custom MCP Server**: Built `mcp_server.py` with Stone Ridge investment tools using `FastMCP`
- **langchain-mcp-adapters**: Used `MultiServerMCPClient` to bridge MCP tools into LangGraph agents
- **Multi-Server Agent**: Combined tools from multiple MCP servers into a single agent

### Key Takeaways

1. **MCP is a protocol, not a product** — any server that implements MCP can be consumed by any MCP-compatible client
2. **`FastMCP` makes server creation trivial** — just decorate Python functions with `@mcp.tool()`
3. **`langchain-mcp-adapters` bridges MCP → LangChain** — MCP tools become regular LangChain tools
4. **Multiple servers compose naturally** — an agent can use tools from many servers simultaneously